[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/Assignment_10_Clustering_and_Heatmaps_in_Colab.ipynb)

[Open Assignment 10 in Google Colab](https://colab.research.google.com/github/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/Assignment_10_Clustering_and_Heatmaps_in_Colab.ipynb) · [View the course master on GitHub](https://github.com/RobBurnap/Bioinformatics-MICR4203-MICR5203/blob/main/notebooks/Assignment_10_Clustering_and_Heatmaps_in_Colab.ipynb)

# Assignment 10 — Clustering and Heatmaps in Colab
**30 points · approximately 105–135 minutes**

**Dataset:** the exact 12-gene × 6-time-point teaching matrix shown in the lecture (genes C–N; 0–10 h). These are illustrative log2 fold changes relative to 0 h, not raw counts or a replicated RNA-seq experiment. Do not normalize these values as counts, log-transform them again, or calculate differential-expression p-values from them. Gene labels have no assigned functions.

**Colab workflow:** Open this notebook with the **Open in Colab** button on Canvas or the link above. Choose **File → Save a copy in Drive** before editing. Work in your personal copy, then choose **Runtime → Run all**. The shared GitHub notebook remains the course master. A standard CPU runtime is sufficient; no Drive mount, external dataset, installation, API key, or GPU is required. Fill the Markdown response cells, rerun after any parameter change, then save and download the executed `.ipynb`. The last cell downloads the results ZIP. Submit both files in Canvas. If the automatic ZIP download is blocked, use Colab's Files panel to download it. Your responses belong in this notebook; no separate Word report is required.

All analysis code is provided. Explain your choices in your own words; disclose any AI assistance and verify its claims against your actual outputs. Save before closing Colab.

**Required hands-on code edits:** In sections B, C, and D, change the lines marked `# EDIT THIS LINE`. The formatted instructions give exact values and rerun steps. Predict the effect before running, compare outputs, and record each trial in the response tables. The supplied plots do not replace these edits.

## Goals and key terms
Connect distance, scaling, k-means, hierarchical clustering and heatmaps to biological interpretation. Key terms: observation, feature, Euclidean distance, correlation distance, centroid, inertia, k, row z-score, average linkage, co-expression.

Lecture anchors: *A shared expression teaching matrix; Scaling changes the question; K-means in the Colab workflow; Average-linkage merges of the same profiles; Heatmap of the shared gene profiles; Choosing k and assessing stability*. The exercise begins with synthetic points, then applies the methods to the lecture matrix.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json, shutil, platform
import sklearn, scipy
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, dendrogram
SEED = 42
logfc = pd.DataFrame([[0, 3, 3.58, 4, 3.58, 3], [0, 1.58, 2, 2, 1.58, 1], [0, 2, 3, 3, 3, 3], [0, 0, 0, -2, -2, -3.32], [0, 1, 1.58, 2, 1.58, 1], [0, -1, -1.6, -2, -1.6, -1], [0, 2, 3, 2, 0, -1], [0, 1, 0, 1, 0, 1], [0, 0, 0, 0, 1.58, 1.58], [0, 1, 1.58, 2, 1.58, 1], [0, -1.6, -2, -2, -1.6, -1], [0, -3, -3.59, -4, -3.59, -3]], index=list("CDEFGHIJKLMN"), columns=["0 h","2 h","4 h","6 h","8 h","10 h"])
assert logfc.shape == (12,6)
assert np.array_equal(logfc.loc['G'], logfc.loc['L'])
result_dir = Path('assignment_10_results')
result_dir.mkdir(exist_ok=True)
logfc.to_csv(result_dir / 'teaching_log2_fold_changes.csv')
print(logfc.to_string())
print('Illustrative log2 fold changes relative to 0 h; no biological replicates.')

### A. Units and orientation — 4 points
Use the displayed matrix and its description.

| Item | Your response |
|---|---|
| What is an observation and what is a feature when clustering genes? | Replace with your answer. |
| What does +2 mean on the original log2 fold-change scale? What does 0 mean? | Replace with your answer. |
| Why are G and L a useful implementation check? | Replace with your answer. |

## B. Synthetic points: choosing k — 6 points
The true generator is known here. Compare k=2, 3, 4; do not assume that an elbow is a statistical proof of a biological class.

First run the baseline cell below unchanged. The new guided experiment immediately after the B response table will ask you to change the scatter and the distance between group centers. Keep the seed fixed so random resampling does not obscure your comparison.

In [ ]:
rng = np.random.default_rng(SEED)  # HOLD FIXED: reproducible random draws
centers = np.array([[-1,1],[1,1],[1,-1]])
variances = [0.05,0.05,0.01]  # Baseline variances; edit the guided cell below instead.
points = np.concatenate([c + rng.normal(size=(25,2))*np.sqrt(v) for c,v in zip(centers,variances)])
rng.shuffle(points)
inertias=[]
for k in range(1,9):
    inertias.append(KMeans(n_clusters=k,n_init=20,random_state=SEED).fit(points).inertia_)
fig,axes=plt.subplots(1,4,figsize=(15,3.5))
axes[0].plot(range(1,9),inertias,'o-'); axes[0].set(xlabel='k',ylabel='Inertia',title='Elbow comparison')
for ax,k in zip(axes[1:],[2,3,4]):
    labels=KMeans(n_clusters=k,n_init=20,random_state=SEED).fit_predict(points)
    ax.scatter(points[:,0],points[:,1],c=labels,cmap='tab10'); ax.set(title=f'k={k}',xlabel='Feature 1',ylabel='Feature 2')
plt.tight_layout(); plt.savefig(result_dir/'synthetic_k_comparison.png',dpi=160); plt.show()
pd.DataFrame({'k':range(1,9),'inertia':inertias}).to_csv(result_dir/'synthetic_inertia.csv',index=False)

### B. Response
Support each answer with your plotted results.

| Item | Your response |
|---|---|
| Choose k for these points and explain using both the scatterplots and inertia. | Replace with your answer. |
| Describe one under-splitting and one over-splitting consequence. | Replace with your answer. |
| Why is recovering synthetic groups easier to assess than discovering biological groups? | Replace with your answer. |

## B continued — Change the artificial data yourself
**This is required code manipulation, not a demonstration to watch.** In the next cell, edit only the two lines marked `# EDIT THIS LINE`. In Python, text after `#` is a comment: edit the number before the comment and leave the variable name and `=` in place.

The **left panel is always a fixed baseline**. The **right panel uses your edited settings**. Both use exactly the same random draws and k=3, so you can isolate the effect of one change. These are synthetic coordinates, not measured gene expression; the separate lecture matrix is unchanged.

1. **Baseline:** leave `noise_multiplier = 1.0` and `separation_multiplier = 1.0`. Run the cell. The panels should match (cluster colors are arbitrary).
2. **More scatter:** predict what will happen, then change only `noise_multiplier = 1.0` to **`noise_multiplier = 3.0`**. Run **this cell only** using its Run button or Shift+Enter. Compare cluster overlap with baseline and record what you see.
3. **Closer centers:** restore **`noise_multiplier = 1.0`**. Predict the effect of bringing group centers closer together. Change only `separation_multiplier = 1.0` to **`separation_multiplier = 0.5`**, then rerun this cell. Record the change in overlap and group recovery.
4. **Restore:** set both numbers back to **1.0** and rerun. Keep the seed, group sizes, baseline variances, and k fixed throughout. Do not change two parameters at once.

`noise_multiplier` multiplies the **standard deviation** of the random deviations: 3× scatter means 9× variance. `separation_multiplier` scales the center coordinates about the origin; 0.5 halves all distances between centers. The cell prints a descriptive recovery score using the known simulated group identities. Adjusted Rand index (ARI) equals 1 for exact agreement up to label permutation; it is a teaching check for these labeled simulations, not a test of biological significance.

In [ ]:
# CHANGE THE NUMBERS ON THESE TWO MARKED LINES, ONE AT A TIME.
noise_multiplier = 1.0       # EDIT THIS LINE: 1.0 -> 3.0 -> restore 1.0
separation_multiplier = 1.0  # EDIT THIS LINE: 1.0 -> 0.5 -> restore 1.0

# HOLD THE REMAINING CODE FIXED during these trials.
from sklearn.metrics import adjusted_rand_score
assert noise_multiplier > 0 and separation_multiplier > 0
experiment_rng = np.random.default_rng(SEED)
experiment_centers = np.array([[-1.,1.],[1.,1.],[1.,-1.]])
experiment_variances = np.array([0.05,0.05,0.01])
# Reuse exactly the same random deviations for baseline and edited points.
unit_deviations = experiment_rng.normal(size=(3,25,2))
base_deviations = unit_deviations * np.sqrt(experiment_variances)[:,None,None]
baseline_points = (experiment_centers[:,None,:] + base_deviations).reshape(-1,2)
edited_points = (experiment_centers[:,None,:]*separation_multiplier + base_deviations*noise_multiplier).reshape(-1,2)
known_groups = np.repeat(np.arange(3),25)
fig,axes = plt.subplots(1,2,figsize=(10,4),sharex=True,sharey=True)
recovery = []
for ax,A,title in zip(axes,[baseline_points,edited_points],['Fixed baseline',f'Your edits: noise={noise_multiplier:g}, separation={separation_multiplier:g}']):
    model = KMeans(n_clusters=3,n_init=20,random_state=SEED).fit(A)
    ari = adjusted_rand_score(known_groups,model.labels_)
    recovery.append(ari)
    ax.scatter(A[:,0],A[:,1],c=model.labels_,cmap='tab10',s=25)
    ax.set(xlabel='Feature 1',ylabel='Feature 2',title=f'{title}\nARI={ari:.3f}')
    ax.set_aspect('equal',adjustable='box')
plt.tight_layout()
trial_tag = f'noise_{noise_multiplier:g}_separation_{separation_multiplier:g}'
plt.savefig(result_dir/f'B_guided_{trial_tag}.png',dpi=160)
plt.show(); plt.close(fig)
print(f'Baseline ARI: {recovery[0]:.3f}; edited ARI: {recovery[1]:.3f}')
trial_B = pd.DataFrame([{'noise_multiplier':noise_multiplier,'separation_multiplier':separation_multiplier,'seed':SEED,'k':3,'baseline_ARI':recovery[0],'edited_ARI':recovery[1]}])
path_B = result_dir/'B_simulation_edit_history.csv'
trial_B.to_csv(path_B,index=False,mode='a' if path_B.exists() else 'w',header=not path_B.exists())
print('Trial recorded. Complete the response row below before your next edit.')

### B continued — Your parameter-edit record (included in B's 6 points)
Make your prediction **before** running each edited setting, then complete the same row with the result. If the runtime restarts, redo the trials so the saved files correspond to your written record.

| Trial: noise, separation | Prediction before running | Observed overlap and edited ARI | Explanation: what changed, and what stayed fixed? |
|---|---|---|---|
| Baseline: 1.0, 1.0 | Predict whether the panels will match. | Replace with your observation and score. | Replace with your explanation. |
| More scatter: 3.0, 1.0 | Replace with your prediction. | Replace with your observation and score. | Replace with your explanation. |
| Closer centers: 1.0, 0.5 | Replace with your prediction. | Replace with your observation and score. | Replace with your explanation. |
| Restored: 1.0, 1.0 | Predict what should return. | Replace with your observation and score. | Explain why keeping the seed fixed made the comparison fair. |

Which edit made cluster recovery harder in this run? Use both visual overlap and the recovery score, and explain why changing scatter and changing separation can have similar consequences. Do not infer that synthetic recovery proves a biological clustering is correct.

## C. Scaling changes the question — 5 points

### Your code edits in this section
1. In the next code cell, find `stretch_factor = 100.0  # EDIT THIS LINE` and change **100.0 to 1.0**. Click the Run button at the left of **that cell only** (or press Shift+Enter). With factor 1, no feature stretching occurs; note the grouping.
2. Change **1.0 back to 100.0** and rerun that same cell. Compare the middle panel with the first and third panels. Keep k=3 and the seed unchanged.
3. Record both trials in C's response table. Leave `stretch_factor = 100.0` for your saved final run.

Do not edit `points` or rerun its generator during this comparison. This section uses the original baseline points, not the separate B experiment. Numeric cluster labels and colors may permute; compare which points stay together.

In [ ]:
stretch_factor = 100.0  # EDIT THIS LINE: try 1.0, then restore 100.0
stretched=points.copy(); stretched[:,0]*=stretch_factor
standardized=StandardScaler().fit_transform(stretched)
fig,axes=plt.subplots(1,3,figsize=(12,3.5))
for ax,A,title in zip(axes,[points,stretched,standardized],['Original',f'Feature 1 × {stretch_factor:g}','Standardized after stretching']):
    labels=KMeans(n_clusters=3,n_init=20,random_state=SEED).fit_predict(A)
    ax.scatter(A[:,0],A[:,1],c=labels,cmap='tab10'); ax.set(title=title,xlabel='Feature 1',ylabel='Feature 2')
plt.tight_layout(); plt.savefig(result_dir/'scaling_comparison.png',dpi=160); plt.show()
trial_C = pd.DataFrame([{'stretch_factor':stretch_factor,'seed':SEED,'k':3}])
path_C = result_dir / 'C_scaling_edit_history.csv'
trial_C.to_csv(path_C,index=False,mode='a' if path_C.exists() else 'w',header=not path_C.exists())
plt.close('all')

### C. Response
Compare group membership, not numeric label colors.

| Item | Your response |
|---|---|
| Explain how the ×100 change alters Euclidean distances. | Replace with your answer. |
| Did standardization restore the original relationships? Cite your plots. | Replace with your answer. |
| When might standardization remove biologically meaningful magnitude information? | Replace with your answer. |

| Required manual trial | What you predicted before running | What changed in the plot? |
|---|---|---|
| stretch_factor = 1.0 | Replace with your prediction. | Replace with your observation. |
| stretch_factor = 100.0 | Replace with your prediction. | Replace with your observation. |


## D. Gene k-means — 5 points
Rows are genes; columns are time points. Row z-scores emphasize relative profile shape and remove between-gene differences in mean and scale. This is a choice, not a universal RNA-seq preprocessing pipeline.

### Your code edits in this section
Find `k_gene = 3  # EDIT THIS LINE` in the next cell. Run it with **3**, predict the consequence of fewer clusters, change the line to **2**, and rerun **that cell only**. Record a concrete merge or membership change in D's response table. Then change it back to **3** and rerun that cell for the saved final analysis. Keep row scaling, `n_init=20`, and `random_state=SEED` fixed. Changing k does not change the input expression values.

In [ ]:
Z = logfc.sub(logfc.mean(axis=1),axis=0).div(logfc.std(axis=1,ddof=0),axis=0)
assert np.allclose(Z.mean(axis=1),0)
assert np.allclose(Z.std(axis=1,ddof=0),1)
k_gene = 3  # EDIT THIS LINE: run 3, change to 2, then restore 3.
gene_model=KMeans(n_clusters=k_gene,n_init=20,random_state=SEED).fit(Z)
clusters=pd.DataFrame({'gene':logfc.index,'cluster':gene_model.labels_+1})
print(clusters.to_string(index=False))
fig,axes=plt.subplots(1,k_gene,figsize=(4*k_gene,3),squeeze=False)
for j,ax in enumerate(axes[0]):
    subset=Z.iloc[np.where(gene_model.labels_==j)[0]]
    for gene,row in subset.iterrows(): ax.plot(logfc.columns,row,alpha=.6,label=gene)
    ax.plot(logfc.columns,gene_model.cluster_centers_[j],'k--',linewidth=2,label='Centroid')
    ax.set(title=f'Cluster {j+1}',ylabel='Within-gene z-score'); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(result_dir/'gene_kmeans.png',dpi=160); plt.show()
clusters.to_csv(result_dir/'gene_clusters.csv',index=False)
Z.to_csv(result_dir/'row_z_scores.csv')
trial_D = clusters.copy()
trial_D['k_gene'] = k_gene
path_D = result_dir / 'D_gene_k_edit_history.csv'
trial_D.to_csv(path_D,index=False,mode='a' if path_D.exists() else 'w',header=not path_D.exists())
plt.close('all')

### D. Response
Manually edit k_gene from 3 to 2, record what changes, then restore 3 as instructed above. Cluster numbers themselves have no biological meaning.

| Item | Your response |
|---|---|
| Predict the effect of k=2 before running; report one observed membership change from k=3 and confirm your final k=3. | Replace with your answer. |
| List one cluster’s genes and describe their time profile. | Replace with your answer. |
| Does co-clustering establish common regulation or function? Explain. | Replace with your answer. |

## E. Hierarchy and heatmap — 6 points
Use correlation distance on the original profiles and **average linkage**. Do not use Ward linkage with correlation distance. Keep time columns chronological.

In [ ]:
distance=np.maximum(pdist(logfc.to_numpy(),metric='correlation'),0)
assert np.isfinite(distance).all()
T=linkage(distance,method='average')
fig=plt.figure(figsize=(10,7))
gs=fig.add_gridspec(1,2,width_ratios=[1,3],wspace=.05)
axd=fig.add_subplot(gs[0]); d=dendrogram(T,labels=logfc.index.tolist(),orientation='left',ax=axd)
axd.set(xlabel='Correlation distance'); axd.set_yticks([])
order=d['leaves']
axh=fig.add_subplot(gs[1]); image=axh.imshow(Z.iloc[order],aspect='auto',origin='lower',cmap='RdBu_r',vmin=-2.5,vmax=2.5)
axh.set_xticks(range(6),logfc.columns); axh.set_yticks(range(12),Z.index[order]); axh.set(xlabel='Time',title='Row-z-score heatmap; correlation/average hierarchy')
fig.colorbar(image,ax=axh,label='Within-gene z-score'); plt.savefig(result_dir/'hierarchical_heatmap.png',dpi=160,bbox_inches='tight'); plt.show()
pd.DataFrame(T,columns=['left','right','distance','members']).to_csv(result_dir/'linkage.csv',index=False)

### E. Response
Use gene names when comparing methods; leaf order may rotate without changing a hierarchy.

| Item | Your response |
|---|---|
| Identify the zero-distance pair and one other close relationship. | Replace with your answer. |
| Explain a difference between k-means partitioning and a dendrogram. | Replace with your answer. |
| Why can red in this heatmap occur at an original value of 0? | Replace with your answer. |
| Why keep time columns in chronological order? | Replace with your answer. |

### F. Interpretation and reproducibility — 4 points
Finish after running the full notebook.

| Item | Your response |
|---|---|
| Give one supported descriptive conclusion and one unsupported biological claim. | Replace with your answer. |
| Record seed, final k, scaling, distance/linkage, and any code changes. | Replace with your answer. |
| Disclose any AI help (or state none) and describe how you checked its output. | Replace with your answer. |

Confirm that you changed the marked lines yourself in B, C, and D. Record any additional edits separately. Before submitting, check that the ZIP includes edit-history CSVs and that your saved notebook contains completed before/after response tables.

In [ ]:
settings = {'assignment': 10, 'seed': SEED, 'data': 'Lecture teaching matrix; illustrative log2 fold changes; no biological replicates', 'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__, 'scipy': scipy.__version__, 'k_gene': k_gene, 'guided_noise_multiplier': noise_multiplier, 'guided_separation_multiplier': separation_multiplier, 'stretch_factor': stretch_factor, 'gene_kmeans_input': 'row z-scores ddof=0; Euclidean', 'hierarchy': 'correlation distance; average linkage; chronological columns'}
(result_dir / 'analysis_record.json').write_text(json.dumps(settings, indent=2))
archive = shutil.make_archive(str(result_dir), 'zip', root_dir=result_dir)
print('Results archive:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Local run: retrieve the ZIP from the working folder.')

## Submission and rubric
Submit the **executed notebook with completed responses** and **assignment_10_results.zip**. Check that plots and response tables are visible in the saved notebook. A 4 + B 6 + C 5 + D 5 + E 6 + F 4 = **30 points**. Credit requires correct results and an explanation tied to those results; screenshots alone are insufficient.

Method references: [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html), [SciPy linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html).

**The total remains 30 points.** Guided manual changes and their before/after explanations are part of the existing sections: B (6), C (5), and D (5). There are no extra-credit points or extra submission files. After completing the trials and response tables, restore the marked lines to their baseline values as instructed and run the notebook from top to bottom once. This refreshes the final ZIP while preserving the edit-history CSVs in the same runtime. If the runtime restarts, the files are lost: redo the required trials before exporting. Repeated runs can add duplicate history rows; this is normal, and the response tables remain the required evidence of your reasoning.